In [ ]:
import os
from pyspark.sql import SparkSession

# Auto-detect runtime environment (Local vs Azure Databricks)
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
STORAGE_ACCOUNT = dbutils.widgets.get("STORAGE_ACCOUNT_NAME") if IS_DATABRICKS else ""
BASE_DATA_PATH = f"abfss://raw-data@{STORAGE_ACCOUNT}.dfs.core.windows.net" if IS_DATABRICKS else "../data"

# Enable Delta Lake support in PySpark
builder = (
    SparkSession.builder.appName("ECommerce_RFM_ML_Pipeline")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

if not IS_DATABRICKS:
    try:
        from delta import configure_spark_with_delta_pip
        spark = configure_spark_with_delta_pip(builder).getOrCreate()
    except ImportError:
        spark = builder.getOrCreate()
else:
    spark = builder.getOrCreate()

curated_input_path = f"{BASE_DATA_PATH}/curated/cleaned_orders.delta"

# Load cleaned Delta file from Notebook 1 (with Parquet fallback)
try:
    df_cleaned = spark.read.format("delta").load(curated_input_path)
except Exception:
    fallback_path = f"{BASE_DATA_PATH}/curated/cleaned_orders.parquet"
    df_cleaned = spark.read.parquet(fallback_path)

df_cleaned.createOrReplaceTempView("cleaned_orders")

print(f"Curated data loaded from ({'Databricks ADLS' if IS_DATABRICKS else 'Local'}): {curated_input_path}")
df_cleaned.show(5)

In [ ]:
rfm_sql = """
WITH max_date_cte AS (
    SELECT MAX(Order_Timestamp) AS max_dataset_date FROM cleaned_orders
)
SELECT 
    c.Customer_ID,
    COUNT(c.Order_ID) AS frequency,
    ROUND(SUM(c.Order_Amount), 2) AS monetary,
    DATEDIFF(m.max_dataset_date, MAX(c.Order_Timestamp)) AS recency,
    AVG(c.Returned_Flag) AS return_rate
FROM cleaned_orders c
CROSS JOIN max_date_cte m
GROUP BY c.Customer_ID, m.max_dataset_date
"""

df_rfm = spark.sql(rfm_sql)
df_rfm.createOrReplaceTempView("rfm_features")

df_rfm.show(5)
print(f"Total Unique Customers: {df_rfm.count()}")

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Define MLflow Experiment Workspace
mlflow.set_experiment("/Shared/Customer_Segmentation_MLOps")

# 2. Convert Spark RFM DataFrame to Pandas
rfm_pd = df_rfm.toPandas()

with mlflow.start_run(run_name="kmeans_rfm_clustering") as run:

    # --- Hyperparameters ---
    k_clusters = 3
    seed_val = 42
    
    mlflow.log_params({
        "k_clusters": k_clusters,
        "seed": seed_val,
        "distance_measure": "euclidean",
        "algorithm": "scikit-learn-kmeans"
    })

    # --- Step 1: Feature Scaling ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(rfm_pd[["recency", "frequency", "monetary"]])

    # --- Step 2: Fit KMeans Model ---
    kmeans = KMeans(
        n_clusters=k_clusters, 
        random_state=seed_val, 
        n_init=10
    )
    cluster_labels = kmeans.fit_predict(X_scaled)

    # --- Step 3: Evaluate Model Quality ---
    score = silhouette_score(X_scaled, cluster_labels)

    # --- Step 4: Log Metrics & Model Artifacts ---
    mlflow.log_metric("silhouette_score", score)
    
    # Log the trained Scikit-Learn model natively to MLflow
    mlflow.sklearn.log_model(
        sk_model=kmeans, 
        artifact_path="kmeans_rfm_model"
    )

    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"Logged Silhouette Score: {score:.4f}")

# --- Step 5: Join Clusters Back to PySpark DataFrame ---
# Append cluster predictions back to your original PySpark DataFrame
rfm_pd["cluster_id"] = cluster_labels
df_clustered = spark.createDataFrame(rfm_pd)

In [ ]:
df_clustered.createOrReplaceTempView("customer_clusters")

cluster_profile_sql = """
SELECT 
    cluster_id,
    COUNT(Customer_ID) AS total_customers,
    ROUND(AVG(recency), 1) AS avg_recency_days,
    ROUND(AVG(frequency), 1) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary_spend,
    ROUND(AVG(return_rate), 2) AS avg_return_rate
FROM customer_clusters
GROUP BY cluster_id
ORDER BY avg_monetary_spend DESC
"""

df_profiles = spark.sql(cluster_profile_sql)
df_profiles.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate mean metrics per cluster in Spark SQL
df_profiles_pd = spark.sql("""
    SELECT 
        cluster_id,
        ROUND(AVG(recency), 1) AS avg_recency,
        ROUND(AVG(frequency), 1) AS avg_frequency,
        ROUND(AVG(monetary), 2) AS avg_monetary
    FROM customer_clusters
    GROUP BY cluster_id
    ORDER BY cluster_id
""").toPandas()

# Plot Average Monetary Spend per Cluster
plt.figure(figsize=(8, 5))
barplot = sns.barplot(
    data=df_profiles_pd,
    x="cluster_id",
    y="avg_monetary",
    palette="viridis"
)
plt.title("Average Monetary Spend by Customer Cluster", fontsize=14)
plt.xlabel("Cluster ID")
plt.ylabel("Average Spend ($)")

# Annotate bars with values
for p in barplot.patches:
    barplot.annotate(
        f"${p.get_height():,.2f}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="center",
        xytext=(0, 9),
        textcoords="offset points",
    )

plt.show()

In [ ]:
# Convert clustered Spark DataFrame to Pandas for local visualization
df_pd = df_clustered.select(
    "recency", "frequency", "monetary", "cluster_id"
).toPandas()

# Set plot style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Monetary vs Recency by Cluster
sns.scatterplot(
    data=df_pd,
    x="recency",
    y="monetary",
    hue="cluster_id",
    palette="viridis",
    s=70,
    ax=axes[0],
)
axes[0].set_title("Customer Segments: Monetary vs Recency", fontsize=14)
axes[0].set_xlabel("Recency (Days Since Last Purchase)")
axes[0].set_ylabel("Monetary Spend ($)")

# Plot 2: Frequency vs Recency by Cluster
sns.scatterplot(
    data=df_pd,
    x="recency",
    y="frequency",
    hue="cluster_id",
    palette="viridis",
    s=70,
    ax=axes[1],
)
axes[1].set_title("Customer Segments: Frequency vs Recency", fontsize=14)
axes[1].set_xlabel("Recency (Days Since Last Purchase)")
axes[1].set_ylabel("Total Frequency (Order Count)")

plt.tight_layout()
plt.show()

In [ ]:
final_output_path = f"{BASE_DATA_PATH}/curated/customer_segments.delta"

# Save clustered RFM features DataFrame as Delta table
df_clustered.drop("raw_features", "scaled_features").write \
    .format("delta") \
    .mode("overwrite") \
    .save(final_output_path)

if IS_DATABRICKS:
    spark.sql(f"CREATE TABLE IF NOT EXISTS default.customer_segments USING DELTA LOCATION '{final_output_path}'")

print(f"Clustered customer dataset saved successfully in Delta format to: {final_output_path}")